<a href="https://colab.research.google.com/github/overgroove/samsung_lecture/blob/main/05_feature_eng_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Feature selection**
- feature engineering중 가장 중요한 단계 중 하나
- 변수의 전처리 과정 중 생성 된 다양한 파생변수에 따라 변수 갯수가 많아진 상황
- 타겟데이터에 유용한 변수만을 선택해준다면 데이터의 노이즈를 줄이고 오버피팅문제를 해결 할 수 있어 모델 퍼포먼스를 끌어올리는데 효과적

## **data loading**

In [ ]:
# 필요모듈 import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

pd.options.display.max_columns = 200
pd.options.display.max_rows = 200

In [ ]:
# ames_transform 데이터 로딩


In [ ]:
# 타겟데이터 분리


In [ ]:
# 학습데이터 분리


전처리 과정중에서 변수가 274개로 늘어난 상황  
통계적 방법보다는 머신러닝 모델을 활용한 변수 선택 방법을 사용

## **1. ML selection**
### **1.1 Lasso coef_활용**
- 선형모델 기반 변수 선택에 자주 사용하는 모델이 Lasso
- Lasso모델은 규제화 모델로 선형모델의 변수가 많아질경우 오버피팅 문제를 해결하기 위해 사용
- 계수값인 $\beta$ 절대값의 합을 비용함수에 추가
- 변수에 해당하는 계수값을 0으로 만들면서 학습
- 결국 모델 결과값에 영향을 주지 않는 변수를 삭제하는 효과를 가짐

In [ ]:
# Lasso import 및 fitting, score 확인


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.100e+10, tolerance: 6.826e+08
  model = cd_fast.enet_coordinate_descent(


0.7418049920807837

In [ ]:
# coef_ threashold 기준 변수 선별


Index(['num_high_skew__MSSubClass', 'num_high_skew__LotFrontage',
       'num_high_skew__LotArea', 'num_high_skew__MasVnrArea',
       'num_high_skew__BsmtFinSF1', 'num_high_skew__BsmtFinSF2',
       'num_high_skew__TotalBsmtSF', 'num_high_skew__1stFlrSF',
       'num_high_skew__LowQualFinSF', 'num_high_skew__GrLivArea',
       ...
       'cat_low_card__SaleType_ConLw', 'cat_low_card__SaleType_New',
       'cat_low_card__SaleType_Oth', 'cat_low_card__SaleType_WD',
       'cat_low_card__SaleCondition_Abnorml',
       'cat_low_card__SaleCondition_AdjLand',
       'cat_low_card__SaleCondition_Alloca',
       'cat_low_card__SaleCondition_Family',
       'cat_low_card__SaleCondition_Normal',
       'cat_low_card__SaleCondition_Partial'],
      dtype='object', length=300)

### **1.2 treebase feature_importances_ 활용**
- DecisionTree 모델을 기반으로 한 ML모델에서 사용가능한 속성값
- 각 트리가 예측/분류 결과값을 내기 위해 트리의 분지에 사용한 변수에 해당 변수로 구분 된 데이터 갯수를 가중치로 normalized 된 값
- 결과를 내기 위해 모델이 중요하게 생각하는 변수의 우선순위

In [ ]:
# treebase model feature importances_ 활용


Index(['num_high_skew__MSSubClass', 'num_high_skew__LotFrontage',
       'num_high_skew__LotArea', 'num_high_skew__MasVnrArea',
       'num_high_skew__BsmtFinSF1', 'num_high_skew__BsmtFinSF2',
       'num_high_skew__TotalBsmtSF', 'num_high_skew__1stFlrSF',
       'num_high_skew__LowQualFinSF', 'num_high_skew__GrLivArea',
       ...
       'cat_low_card__SaleType_ConLI', 'cat_low_card__SaleType_ConLw',
       'cat_low_card__SaleType_New', 'cat_low_card__SaleType_Oth',
       'cat_low_card__SaleType_WD', 'cat_low_card__SaleCondition_Abnorml',
       'cat_low_card__SaleCondition_Alloca',
       'cat_low_card__SaleCondition_Family',
       'cat_low_card__SaleCondition_Normal',
       'cat_low_card__SaleCondition_Partial'],
      dtype='object', length=1792)

### **1.3 permutation_importance(치환중요도)활용**
- 특정 변수의 샘플값을 무작위로 섞었을 때 모델의 성능이 얼만큼 감소하는지를 기준으로 변수의 중요도를 판단하는 기법
- 학습 된 모델에 테스트데이터 전달하여 기준점수(평가지표 스코어)측정
- 특정 변수값(샘플)을 무작위로 셔플하여 데이터에 노이즈 추가, 이 작업을 통해 타겟데이터와 해당 변수와의 관계가 완전히 파괴
- 뒤섞인 데이터를 모델에 전달하여 평가지표 재측정
- (원래점수 - 뒤섞인데이터 점수)가 높으면 높을수록 해당 변수에 의존하고 있다는 의미로 높은 중요도로 판단한다.

In [ ]:
# sklearn.inspection permutation_importance


In [ ]:
%%time
# 모델 생성 및 학습


CPU times: user 10min 58s, sys: 10.6 s, total: 11min 9s
Wall time: 11min 14s


In [ ]:
# result 기준 변수 중요도 순으로 변수 정리


,feature,importance
20,num_low_skew__OverallQual,0.525307
9,num_high_skew__GrLivArea,0.130010
6,num_high_skew__TotalBsmtSF,0.021065
7,num_high_skew__1stFlrSF,0.015439
4,num_high_skew__BsmtFinSF1,0.009928
...,...,...
682,cat_high_card__Exterior1st_129000,-0.000364
865,cat_high_card__Exterior1st_190000,-0.000409
36,num_low_skew__YrSold,-0.000445
878,cat_high_card__Exterior1st_197000,-0.000520


In [ ]:
# 중요도 기준 threshold 적용 변수 선택


,num_low_skew__OverallQual,num_high_skew__GrLivArea,num_high_skew__TotalBsmtSF,num_high_skew__1stFlrSF,num_high_skew__BsmtFinSF1,num_low_skew__GarageCars,num_high_skew__LotArea,num_low_skew__GarageArea,num_low_skew__TotRmsAbvGrd,num_low_skew__2ndFlrSF,num_low_skew__YearRemodAdd,num_low_skew__OverallCond,num_low_skew__YearBuilt,num_low_skew__Fireplaces,cat_high_card__Neighborhood_280000,num_high_skew__MasVnrArea,cat_low_card__GarageFinish_Unf,num_low_skew__FullBath,cat_low_card__CentralAir_Y
0,0.636031,0.453551,0.368189,1.301065,-1.420419,0.296063,0.247419,0.253813,0.279234,-0.802294,-0.711644,0.392266,-0.032378,0.594240,0.000000,1.303490,1.0,0.767840,1.0
1,-0.823057,-0.904483,-0.387725,-0.114095,0.689403,0.296063,-1.723115,0.229814,-0.948194,-0.802294,0.976164,-0.508624,1.106434,-0.943262,0.000000,0.821102,0.0,-1.069941,1.0
2,0.636031,0.201283,-0.106255,-1.177818,-0.343515,0.296063,-0.103809,0.656997,0.279234,0.973582,1.169056,-0.508624,1.236584,-0.943262,0.000000,-0.845461,1.0,0.767840,1.0
3,2.095118,0.828959,0.698197,1.692265,-1.420419,1.665189,0.465688,1.492165,0.892948,-0.802294,1.024387,-0.508624,1.138972,0.594240,0.000000,1.321431,0.0,0.767840,1.0
4,0.636031,0.415668,-0.058123,-0.929013,0.572161,0.296063,0.295441,0.349809,-0.334480,1.089252,0.542156,0.392266,0.813597,0.594240,0.000000,-0.845461,0.0,0.767840,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,-0.823057,-0.949810,0.184570,-0.161329,0.835078,-1.073062,-0.208378,-0.811747,-0.334480,-0.802294,0.349263,0.392266,-0.422828,-0.943262,0.000000,-0.845461,0.0,-1.069941,1.0
1456,-2.282144,-0.658883,0.267828,0.141836,0.436075,-1.073062,-0.852266,-1.234131,-0.334480,-0.802294,0.976164,2.194045,-0.520441,-0.943262,0.000000,-0.845461,1.0,-1.069941,1.0
1457,0.636031,0.422921,-0.043009,-0.948056,-1.420419,0.296063,-0.220232,0.407407,0.279234,1.109665,1.072610,0.392266,1.171509,-0.943262,0.000000,0.918381,0.0,0.767840,1.0
1458,-0.093513,-0.574538,-0.193325,-1.525091,-1.420419,0.296063,0.573592,0.244213,0.279234,0.358943,0.735048,2.194045,-1.659253,-0.943262,0.000000,-0.845461,1.0,-1.069941,1.0


## 1.4 **SHAP value**
- **SHAP(SHapley Additive exPlanations)** 은 개별 데이터의 예측값에 대해 각 피처가 어떤 영향을 주었는지 수학적으로 계산하는 기법.
- 단순히 "어떤 피처가 중요한가"를 넘어, 그 피처가 예측값을 얼마나(+/-) 변화시켰는지를 설명
- SHAP은 게임 이론의 Shapley Value를 머신러닝에 이식  
여러 명의 선수(피처)가 팀(모델)을 이뤄 점수(예측값)를 냈을 때, 각 선수의 공헌도를 공정하게 배분하는 방식입니다.

$$f(x) = BaseValue + \sum_{i=1}^{M} \phi_i$$

> BaseValue ($E[f(X)]$): 모델에 아무런 정보를 주지 않았을 때의 평균 예측값입니다.  
> SHAP Value ($\phi_i$): 특정 피처 $i$가 추가됨으로써 베이스라인 대비 예측값이 변한 정도  
> Final Prediction ($f(x)$): 모든 피처의 SHAP Value를 베이스라인에 더하면 최종 예측값이 됩니다.

- 딥러닝 모델에도 적용 가능하며 최근 머신러닝 및 딥러닝 프로젝트에는 필수적으로 사용하는 하는 요소입니다.
- 모델이 복잡해짐에 따라 변수와 변수간 관계 분석이 어려워짐으로 인해 모델 출력결과값을 기반으로 한 변수중요도를 파악합니다.

In [ ]:
# 모듈 import


### **1.4.1 summary plot**
- 변수에 따른 모델의 전체적인 경향성을 파악하는 목적으로 사용
- 피처 중요도와 영향력의 방향(Positive/Negative)을 동시에 확인 가능함.

In [ ]:
# summary plot


### **1.4.1 shap value feature selection**
- 각 변수에 속한 샘플데이터의 shap value 평균값으로 변수 선별
- +/- 영향도를 모두 고려할 수 있도록 절대값 평균을 사용

In [ ]:
# selection
